In [ ]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import json
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from pydantic import BaseModel
from collections import defaultdict

from langchain_community.retrievers import BM25Retriever

/home/rajch/Document/Self rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
load_dotenv(dotenv_path=".env")
# print("API KEY:", os.getenv("OPENAI_API_KEY"))

In [3]:
# Extract the contents from the documents
def document_partitioning(file_path:str):
    """Extrating Structured data from the Unstructured Documents"""

    print(f"Partitioning the contents of the file from {file_path}")

    elements = partition_pdf(
        filename = file_path,
        strategy = "hi_res",
        infer_table_structure=True,
        extract_image_block_types=['Image'],
        extract_image_block_to_payload=True
    )
    
    print(f"Partitioned elements : \n {elements}")
    
    return elements

In [4]:
# elements = document_partitioning("Document/attention-is-all-you-need-Paper.pdf")

In [5]:
## Chucking the extracted sturctured elements using title
def chunking_by_title(elements):
    
    print(f"\n\nChunking the elements using title method.\n")

    chunks = chunk_by_title(
        elements= elements,
        max_characters = 3000,
        new_after_n_chars = 2500,
        combine_text_under_n_chars = 500
    )
    
    print(f"Loading the chunk size : {len(chunks)}\n")

    return chunks

In [6]:
# chunks = chunking_by_title(elements)
# # chunks

In [7]:
# type(chunks[0])

In [8]:
# chunks[0].to_dict()

In [9]:
# chunks[5].metadata.orig_elements[7].to_dict()

In [10]:
def seperate_using_types(chunks):
    
    content_data={
        'text': chunks.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    if hasattr(chunks, 'metadata') and hasattr(chunks.metadata, 'orig_elements'):
        for elements in chunks.metadata.orig_elements:
            elements_type = type(elements).__name__
            
            if elements_type == 'Table':
                content_data['types'].append('table')
                table_content = getattr(elements.metadata, 'text_as_html', elements.text)
                content_data['tables'].append(table_content)
                
            elif elements_type == 'Image':
                if hasattr(chunks, 'metadata') and  hasattr(chunks.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(elements.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))
       
    return content_data


In [11]:
def summarize_content(text:str, tables:list[str], images:list[str])->str:
    """
    Summarize the contents of the table and images.
    """
    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        #Creating a prompt for the llm
        prompt_text = f"""You are to create a searchable description for document content retrival
        
            CONTENT TO ANALYZE:
            TEXT:
            {text}
            
        """

        if tables:
            prompt_text += "Tables \n"
            
            for i, table in enumerate(tables):
               prompt_text += f"Table {i+1}: \n {table}\n\n" 
               
               prompt_text += """
               YOUR TASK:
               Generate a comprehensive, searchable description that covers:
               
               1. Key facts, numbers, and data points from text and tables
               2. Main topics and concepts discussed
               3. Questions this content could answer
               4. Visual content analysis (charts, diagrams, patterns in images)
               5. Alternative search terms users might use
               
               Make it detailed and serachable - prioritize findability over brevity
               
               SEARCHABLE DESCRIPTION:
               """

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
            
        message = HumanMessage(content=message_content)
            
        response = llm.invoke([message])
        
        return response.content
    except Exception as e:
        print(f" AI summary failed: {e}")
        #Fallback to simple summary
        summary = f"{text[:300]}"
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary    

In [12]:
def summarize_chunks(chunks):
    """
    Summarize the chunks which contain text, images, tables 
    """
    
    langchain_document = []
    
    # total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        # current_chunk = i+1
        
        #Analyze the chunk 
        analyze_chunk = seperate_using_types(chunk)
            
        if analyze_chunk['tables'] or analyze_chunk['images']:
            
            print(f"Creating summary for the tables and images")
            
            try:
                enhanced_summary = summarize_content(
                    analyze_chunk['text'],
                    analyze_chunk['tables'],
                    analyze_chunk['images']
                )
                print("AI summary was completed successfully.")

            except Exception as e:
                print(f"Summarizing failed {e}")
        
                enhanced_summary = analyze_chunk['text']

        else:
            print("No tables or image are found.")
            enhanced_summary = analyze_chunk['text']


        # Creating Langchain document
        doc = Document(
            page_content = enhanced_summary,
            metadata = {
                "original_content": json.dumps({
                    "raw_text": analyze_chunk['text'],
                    "tables_html": analyze_chunk['tables'],
                    "images_base64": analyze_chunk['images']
                })
            }
        )
        
        langchain_document.append(doc)
    
    print(f"Created langchain document of chunks size of {len(langchain_document)}.")

    return langchain_document

In [ ]:
doc = summarize_chunks(chunks)

In [ ]:
doc[0]

In [ ]:
# To Visualize the langchain document, im exporting it to a json file.
def export_docs_to_json(chunks, filename="chunks_export.json"):
    
    export_data=[]
    
    for i,chunk in enumerate(chunks):
        chunk_data={
            "chunk_id":i+1,
            "enhanced_content_summary": chunk.page_content,
            "metadata":{
                "original_content": json.loads(chunk.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    #create and save the json file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print("Json file created!")
    

export_docs_to_json(doc)

In [ ]:
def convert_to_embeddings(chunks, persistant_directory="db/chroma.db"):
    """Convert the chunks into vector representation and store it in chroma.db"""
    
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    embeddings = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persistant_directory,
        collection_metadata={"hnsw:space":"cosine"}
    )
    
    print("Vector database creation completed!")
    
    return embeddings

vec_doc = convert_to_embeddings(doc)

In [ ]:
# def run_complete_ingestion_pipeline(pdf_path: str):
#     """Run the complete RAG ingestion pipeline"""
#     print("Startng RAG ingestion pipeline")
#     print("="*50)
    
#     #Step 1: Partition
#     elements = document_partitioning(pdf_path)
    
#     #Step 2: Chunk
#     chunks = chunking_by_title(elements)
    
#     #Step 3: AI Summarisation
#     summarizechunks = summarize_chunks(chunks)
    
#     #Step 4: Vector Store
#     db = convert_to_embeddings(summarizechunks, persistant_directory="dbv1/chroma.db")
    
#     print("Pipeline completed successfully")
    
#     return db

In [ ]:
# db = run_complete_ingestion_pipeline("Document/attention-is-all-you-need-Paper.pdf")

In [ ]:
## This is a simple retrival technique using vector search. 



# query="what is a transformer?"
# retriver = db.as_retriever(search_kwargs={"k":3})
# result_chunks=retriver.invoke(query)

# export_docs_to_json(result_chunks,"rag_result.json")

# def generate_final_answer(result_chunks, query):
#     """Generate final answer using multimodel content"""
    
#     try:
        
#         #Initiate LLM (needs vision model for images)
#         llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
#         #Build the text prompt
#         prompt_text = f"""Based on the following documents, please answer the question {query}
        
#         CONTENT TO ANALYZE:
#         """
        
#         for i, chunk in enumerate(result_chunks):
#             prompt_text += f"------Document {i+1} ----\n"
            
#             if "original_content" in chunk.metadata:
#                 original_data = json.loads(chunk.metadata["original_content"])
                
#                 # Add raw text
#                 raw_text = original_data.get("raw_text", "")
#                 if raw_text:
#                     prompt_text += f"Text: \n{raw_text}\n\n"
                    
                    
#                 # Add tables as HTML
#                 tables_html = original_data.get("tables_html", "")
#                 if tables_html:
#                     prompt_text += f"Tables: \n"
#                     for j, tables in enumerate(tables_html):
#                         prompt_text += f"Table {j+1}: \n{tables}\n\n"
                        
#             prompt_text += "\n"
            
#         prompt_text += """
#         Please provide a clear, comprehensive answer using the text, tables, and images above. If the document
#         don't contain sufficient information to answer the question, say "I don't have information to answer the 
#         question based on the provided documents."
        
#         ANSWER:
#         """
        
#         #Build message content starting with text
#         message_content = [{"type": "text", "text":prompt_text}]
        
#         # Add all images from all chunks
#         for chunk in result_chunks:
#             if "original_content" in chunk.metadata:
#                 original_chunk = json.loads(chunk.metadata["original_content"])
#                 images_base64 = original_chunk.get("images_base64", []) 
                
#                 for images_base64 in images_base64:
#                     message_content.append({
#                         "type": "image_url",
#                         "image_url": {"url": f"data:image/jpeg;base64,{images_base64}"}
#                     })
                    
#         # Send to AI and get response
#         message = HumanMessage(content=message_content)
#         response = llm.invoke([message])
        
#         return response.content
    
#     except Exception as e:
#         print(f"Answer generation failed: {e}")
#         return "Sorry, I encountered an error while generating the answer"

# #Usage
# final_answer = generate_final_answer(result_chunks, query)
# print(final_answer)

In [ ]:
#Pydantic model for structured output
class Query_Variation(BaseModel):
    queries: list[str]
    
    
    
#Original Query
original_query = "How many attention heads does the transformer use, and what is the dimention of each head?"
print(f"Original query:'{original_query}'")



# Generate Multiple query variation of the same query
llm = ChatOpenAI(model="gpt-4o", temperature=0)

llm_with_tools = llm.with_structured_output(Query_Variation)

prompt = f"""Generate 3 different variation of the query that would help
retrieve relevant documents.

Original query: {original_query}

Return 3 alternative queries that rephrase or approach the same question with 
different angles.
"""

response = llm_with_tools.invoke(prompt)
query_variation = response.queries

print("Generate Query Variations")
for i, variation in enumerate(query_variation, 1):
    print(f"{i}, {variation}")
    
print("\n"+"="*60)

Generate Query Variations
1, What is the number of attention heads utilized by the transformer, and what are the dimensions of each head?
2, Can you provide details on the count of attention heads in a transformer and the size of each head?
3, How many attention heads are implemented in a transformer model, and what is the dimensionality of each head?



In [29]:
# Search with each query variation and store the results

#Semantic Search / Dense Retrieval
vec_retriever = db.as_retriever(search_kwargs={'k':5})

#Keyword Search / Sparse Retrieval
bm25_retriever = BM25Retriever.from_documents(summarizechunks)
bm25_retriever.k = 3

dense_retrieval_results = []
sparse_retrieval_results = []

for i, query in enumerate(query_variation, 1):
    print(f"\n ________ Results for query {i}: {query}_______")
    
    dense_docs = vec_retriever.invoke(query)
    dense_retrieval_results.append(dense_docs)
    
    sparse_docs = bm25_retriever.invoke(query)
    sparse_retrieval_results.append(sparse_docs)
    
    print(f"Vector search Retrieved {len(dense_docs)} documents:\n")
    print(f"Keyword search Retrieved {len(sparse_docs)} documents:\n")
    
    for j, doc in enumerate(dense_docs, i):
        print(f"Dense Document {j}:")
        print(f"{doc.page_content[:150]}.......\n")
    
    print("-"*50)
    print("-"*50)
    
    for j, doc in enumerate(sparse_docs, i):
        print(f"Sparse Document {j}:")
        print(f"{doc.page_content[:150]}.......\n")
    
    print("-"*50)
    print("-"*50)
    
print("\n" + "=" * 60)
print("Multi-Query Retrival completed")

NameError: name 'summarizechunks' is not defined

In [ ]:
#Reciprocal Rank Fusion
def reciprocal_rank_fusion(chunk_list, k=60, verbose=True):
    if verbose:
        print("*"*50)
        print("Applying Reciprocal Rank Fusion")
        print("*"*50)
    
    #Data Structures for RRF calculation
    rrf_scores = defaultdict(float)
    unique_chunk_objects = {}
    
    chunk_id_map={}
    chunk_no = 1
       
    for chunk_id, chunks in enumerate(chunk_list, 1): 
        
        for position, chunk in enumerate(chunks, 1):
            chunk_content = chunk.page_content
            
            if chunk_content not in chunk_id_map:
                chunk_id_map[chunk_content] = f"Chunk_{chunk_no}"
                chunk_no += 1
                
            chunk_id = chunk_id_map[chunk_content]
            
            unique_chunk_objects[chunk_content] = chunk
            
            position_score = 1/(k+position)
            
            rrf_scores[chunk_content] += position_score
            
            if verbose:
                print("Completed ranking the chunks!")
                
    
        #Sorting the rrf list in ascending or decending order
    sorting_chunk = sorted(
        [(unique_chunk_objects[chunk_content], scores) for chunk_content, scores in rrf_scores.items()],
        key=lambda x: x[1],
        reverse=True
    )
        
    return sorting_chunk
                    
reciprocal_rank_fusion(all_retrival_results)

**************************************************
Applying Reciprocal Rank Fusion
**************************************************
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!
Completed ranking the chunks!


[(Document(id='525fea08-d5f0-4970-ab6a-98b58c5b5286', metadata={'original_content': '{"raw_text": "3.2.3 Applications of Attention in our Model\\n\\nThe Transformer uses multi-head attention in three different ways:\\n\\n\\u2022 In \\"encoder-decoder attention\\" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [31, 2, 8].\\n\\n\\u2022 The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.\\n\\n\\u2022 Similarly, self-attention layers in the decoder allow each position in the decoder to at